# PHASE 2 — Seed Variance (1 tab = 1 seed)

**Chạy song song 3 tab Colab riêng biệt:**
- Tab A → `SEED = 42`
- Tab B → `SEED = 43`  
- Tab C → `SEED = 44`

**Thay đổi duy nhất:** dòng `SEED = ...` đầu Cell 1.  
Thứ tự: **Cell 1 → Cell 2 → Cell 3**. Không thay đổi gì khác.

Thời gian mỗi tab: ~2 giờ (T4, 35 epoch, two-stage).

In [ ]:
# ==============================================================================
# Cell 1 — Setup: Đặt seed, clone repo, cài deps, chuẩn bị dataset
# ĐỔI SEED = 42 / 43 / 44 tuỳ tab Colab
# ==============================================================================
SEED = 42  # <-- ĐỔI Ở ĐÂY
RUN_DIR = f'/content/phase2_seed{SEED}'

!nvidia-smi

!rm -rf /content/SAGE_LITE
!git clone https://github.com/bach0823/SAGE_LITE.git /content/SAGE_LITE
%cd /content/SAGE_LITE
!git rev-parse HEAD

!pip install -q timm albumentations scipy

!python prepare_data/prepare_crack500.py

import os
os.makedirs(RUN_DIR, exist_ok=True)
print(f'Ready. SEED={SEED}  RUN_DIR={RUN_DIR}')

In [ ]:
# ==============================================================================
# Cell 2 — Training (1 lệnh duy nhất)
# Đừng thay đổi gì trong cell này.
# ==============================================================================
%cd /content/SAGE_LITE

!python scripts/train_crack.py \
    --config configs/sage_b2_crack500.yaml \
    --seed $SEED \
    --data-root /content/dataset/Crack500 \
    --output-dir $RUN_DIR \
    --num-workers 2 \
    2>&1 | tee $RUN_DIR/train.log

print('Training complete.')

In [ ]:
# ==============================================================================
# Cell 3 — Val Eval + Routing Diagnostics + Zip + Download
# Chạy ngay sau Cell 2 khi training xong.
# ==============================================================================
%cd /content/SAGE_LITE
from google.colab import files
import shutil, os

# 1. Val evaluation (KHÔNG chạm test)
!python scripts/evaluate_crack_official.py \
    --config configs/sage_b2_crack500.yaml \
    --checkpoint $RUN_DIR/best_model_b2_global.pth \
    --data-root /content/dataset/Crack500 \
    --split val \
    --diagnostic \
    --output_json $RUN_DIR/val_result.json

# 2. Routing diagnostics
!python tools/analyze_routing.py \
    --config configs/sage_b2_crack500.yaml \
    --checkpoint $RUN_DIR/best_model_b2_global.pth \
    --output_dir $RUN_DIR/routing_diagnostics \
    --split val \
    --data_root /content/dataset/Crack500

# 3. Zip toàn bộ run (best_model + last_model + logs + diagnostics)
zip_base = f'/content/phase2_seed{SEED}_full'
shutil.make_archive(zip_base, 'zip', '/content', f'phase2_seed{SEED}')
zip_file = zip_base + '.zip'
print(f'Zip: {os.path.getsize(zip_file)/1e6:.1f} MB')

# 4. Download về máy
files.download(zip_file)